In [1]:
### Enabling TF32 for A100 Optimizing
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import os, math, time
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from tqdm import tqdm

In [3]:
MODEL_ID        = "Salesforce/codegen-350M-mono"
MAX_LEN         = 512
SAVE_EVERY = 200
# ── Batch / gradient accumulation ────────────────────────────
# Effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS
# Tune BATCH_SIZE down if you hit OOM.
BATCH_SIZE          = 8
GRAD_ACCUM_STEPS    = 24    # effective batch = 192
EPOCHS              = 2
LR                  = 2e-5
WARMUP_RATIO        = 0.05
GRAD_CLIP           = 1.0
EARLY_STOP_LOSS     = 1.2
LOG_EVERY           = 50     # steps (post-accumulation)

DRIVE_ROOT  = "/content/drive/MyDrive/codegen_cp"
TOK_DIR     = f"{DRIVE_ROOT}/tokenizer"
DATA_PATH   = f"{DRIVE_ROOT}/processed/tokenized_data.pt"
CKPT_BEST   = f"{DRIVE_ROOT}/checkpoints/best"
CKPT_LAST   = f"{DRIVE_ROOT}/checkpoints/last"

In [4]:
assert torch.cuda.is_available(), "No GPU found — switch to A100 runtime!"
device  = torch.device("cuda")
DTYPE   = torch.bfloat16    # A100 has native BF16 support (faster + stable)
print(f"Device : {torch.cuda.get_device_name(0)}")
print(f"dtype  : {DTYPE}")

Device : NVIDIA A100-SXM4-40GB
dtype  : torch.bfloat16


In [5]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(CKPT_BEST, exist_ok=True)
os.makedirs(CKPT_LAST, exist_ok=True)

Mounted at /content/drive


In [6]:
tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
print(f"Tokenizer loaded | vocab size: {len(tokenizer)}")

Tokenizer loaded | vocab size: 50296


In [7]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(device)

# Resize embeddings if [PAD] was added in Notebook 1
if len(tokenizer) != model.config.vocab_size:
    model.resize_token_embeddings(len(tokenizer))
    print(f"Embeddings resized to {len(tokenizer)}")

# torch.compile for kernel fusion (PyTorch 2.x, available on Colab A100)
model = torch.compile(model, mode= "default")
print("Model compiled with torch.compile()")
model.train()

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings resized to 50296


model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

Model compiled with torch.compile()


OptimizedModule(
  (_orig_mod): CodeGenForCausalLM(
    (transformer): CodeGenModel(
      (wte): Embedding(50296, 1024)
      (drop): Dropout(p=0.0, inplace=False)
      (h): ModuleList(
        (0-19): 20 x CodeGenBlock(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): CodeGenAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
          )
          (mlp): CodeGenMLP(
            (fc_in): Linear(in_features=1024, out_features=4096, bias=True)
            (fc_out): Linear(in_features=4096, out_features=1024, bias=True)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )


In [9]:
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [10]:
#class CodeContestDataset(Dataset):
#    def __init__(self, data):
#        self.data = data

 #   def __len__(self):
 #       return len(self.data)

  #  def __getitem__(self, idx):
  #      item = self.data[idx]
   #     return {
   #         "input_ids":      torch.tensor(item["input_ids"],      dtype=torch.long),
    #        "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
    #        "labels":         torch.tensor(item["labels"],         dtype=torch.long),
    #    }

class CodeContestDataset(Dataset):
    def __init__(self, data):
        self.input_ids = torch.stack([torch.tensor(d["input_ids"], dtype=torch.long) for d in data])
        self.attention_mask = torch.stack([torch.tensor(d["attention_mask"], dtype=torch.long) for d in data])
        self.labels = torch.stack([torch.tensor(d["labels"], dtype=torch.long) for d in data])

    def __len__(self):
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

tokenized_data = torch.load(DATA_PATH, weights_only=False)
print(f"Loaded {len(tokenized_data)} tokenized examples")

dataset = CodeContestDataset(tokenized_data)
loader  = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,   # avoid worker re-spawn overhead per epoch
    prefetch_factor=2,
)
print(f"DataLoader ready | {len(loader)} batches/epoch")

Loaded 290300 tokenized examples
DataLoader ready | 36288 batches/epoch


In [11]:
# ── 8. Optimizer & scheduler ────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
    fused=True,               # A100: fused AdamW is ~2× faster
)

optimizer_steps = math.ceil(len(loader) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps    = int(WARMUP_RATIO * optimizer_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=optimizer_steps,
)

print("\n--- Training config ---")
print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Optimizer steps      : {optimizer_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"LR                   : {LR}")
print(f"Epochs               : {EPOCHS}")


--- Training config ---
Effective batch size : 192
Optimizer steps      : 3024
Warmup steps         : 151
LR                   : 2e-05
Epochs               : 2


In [12]:
def compute_loss(batch):
    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

    # Signal the start of a new step to torch.compile's CUDA Graphs
    torch.compiler.cudagraph_mark_step_begin()

    if (batch["labels"] == -100).all():
        return None   # skip fully-masked batches

    # Mixed-precision forward pass
    with torch.autocast(device_type="cuda", dtype=DTYPE):
        out = model(**batch)
    return out.loss

In [13]:
best_loss   = float("inf")
early_stop  = False
global_step = 0       # counts optimizer (post-accumulation) steps

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*60}")

    epoch_loss  = 0.0
    epoch_count = 0
    accum_loss  = 0.0
    t0 = time.time()

    optimizer.zero_grad()

    for local_step, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch+1}")):

        loss = compute_loss(batch)
        if loss is None:
            continue

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"NaN/Inf at local_step {local_step}, skipping")
            continue

        # Scale loss for gradient accumulation
        scaled_loss = loss / GRAD_ACCUM_STEPS
        scaled_loss.backward()

        accum_loss  += loss.item()
        epoch_loss  += loss.item()
        epoch_count += 1

        # Optimizer step every GRAD_ACCUM_STEPS batches
        is_last_batch = (local_step + 1 == len(loader))
        if (local_step + 1) % GRAD_ACCUM_STEPS == 0 or is_last_batch:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            avg_accum = accum_loss / GRAD_ACCUM_STEPS
            accum_loss = 0.0

            if avg_accum < best_loss:
                best_loss = avg_accum
                # Save best model
                model.save_pretrained(CKPT_BEST)
                tokenizer.save_pretrained(CKPT_BEST)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                print(
                    f"  step {global_step:5d} | "
                    f"loss {avg_accum:.4f} | "
                    f"best {best_loss:.4f} | "
                    f"lr {scheduler.get_last_lr()[0]:.2e} | "
                    f"{elapsed:.0f}s elapsed"
                )
                t0 = time.time()

            # Early stopping on smoothed loss
            if avg_accum < EARLY_STOP_LOSS:
                print(
                    f"\nEarly stop at global_step {global_step} — "
                    f"loss {avg_accum:.4f} < target {EARLY_STOP_LOSS}"
                )
                early_stop = True
                break

    if early_stop:
        break

    # ── End of epoch: save checkpoint ───────────────────────
    avg_epoch_loss = epoch_loss / max(epoch_count, 1)
    print(f"\nEpoch {epoch+1} complete | avg loss: {avg_epoch_loss:.4f}")
    model.save_pretrained(CKPT_LAST)
    tokenizer.save_pretrained(CKPT_LAST)
    torch.save({
        "epoch":               epoch + 1,
        "global_step":         global_step,
        "best_loss":           best_loss,
        "optimizer_state":     optimizer.state_dict(),
        "scheduler_state":     scheduler.state_dict(),
    }, f"{CKPT_LAST}/training_state.pt")
    print(f"Checkpoint saved → {CKPT_LAST}")

print(f"\n{'='*60}")
print(f"Training complete.")
print(f"Best loss achieved : {best_loss:.4f}")
print(f"Best model saved   : {CKPT_BEST}")
print(f"{'='*60}")


Epoch 1/2


Epoch 1:   0%|          | 23/36288 [01:32<3:35:37,  2.80it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1:   0%|          | 47/36288 [02:22<1:37:08,  6.22it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1:  27%|██▋       | 9958/36288 [21:28<36:02, 12.17it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1:  27%|██▋       | 9959/36288 [21:30<56:50,  7.72it/s]


Early stop at global_step 415 — loss 1.1938 < target 1.2

Training complete.
Best loss achieved : 1.1938
Best model saved   : /content/drive/MyDrive/codegen_cp/checkpoints/best
